In [ ]:
import os
from pathlib import Path

import pandas as pd

DATA_DIR = Path(os.getenv('HOME_CREDIT_DATA_DIR', '.')).expanduser().resolve()

TABLE_PATHS = {
    'application_train': DATA_DIR / 'application_train.csv',
    'application_test': DATA_DIR / 'application_test.csv',
    'bureau': DATA_DIR / 'bureau.csv',
    'bureau_balance': DATA_DIR / 'bureau_balance.csv',
    'previous_application': DATA_DIR / 'previous_application.csv',
    'pos_cash_balance': DATA_DIR / 'POS_CASH_balance.csv',
    'credit_card_balance': DATA_DIR / 'credit_card_balance.csv',
    'installments_payments': DATA_DIR / 'installments_payments.csv',
    'sample_submission': DATA_DIR / 'sample_submission.csv',
    'columns_description': DATA_DIR / 'HomeCredit_columns_description.csv',
}

missing_files = [str(path) for path in TABLE_PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('\n'.join(missing_files))

# Load the small baseline tables immediately; load larger relational tables on demand.
application_train = pd.read_csv(TABLE_PATHS['application_train'])
application_test = pd.read_csv(TABLE_PATHS['application_test'])
sample_submission = pd.read_csv(TABLE_PATHS['sample_submission'])
columns_description = pd.read_csv(TABLE_PATHS['columns_description'], encoding='latin1')

def load_table(name, **read_csv_kwargs):
    return pd.read_csv(TABLE_PATHS[name], **read_csv_kwargs)

application_train.shape, application_test.shape


# Home Credit TC1 Human Baseline

Author: annotator_a

This notebook records a manual TC1 baseline pass over the Home Credit application table and selected relational history tables.

## Manual Sections Covered

- Application cleanup
- Application categorical one-hot encoding
- Bureau balance severity rollup and bureau overdue aggregation
- POS / credit-card / installments aggregation
- Aggregate joins back to the main application table

In [ ]:
# Application cleanup and selected application-table one-hot encoding.
app_train = application_train.loc[application_train['CODE_GENDER'] != 'XNA'].copy()
app_test = application_test.loc[application_test['CODE_GENDER'] != 'XNA'].copy()

app_train['_split'] = 'train'
app_test['_split'] = 'test'
app_test['TARGET'] = pd.NA

application_cats = [
    'CODE_GENDER',
    'NAME_FAMILY_STATUS',
    'NAME_EDUCATION_TYPE',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',
]

app_all = pd.concat([app_train, app_test], ignore_index=True, sort=False)
app_all = pd.get_dummies(app_all, columns=application_cats, dummy_na=False)

def flatten_groupby_result(df, prefix):
    out = df.copy()
    out.columns = [
        f"{prefix}{'_'.join(str(part).upper() for part in col if part)}"
        for col in out.columns.to_flat_index()
    ]
    return out.reset_index()

app_all.shape


In [ ]:
# Bureau-balance severity mapping, monthly-status rollup, and join back to bureau.
bureau = load_table(
    'bureau',
    usecols=[
        'SK_ID_CURR',
        'SK_ID_BUREAU',
        'CREDIT_DAY_OVERDUE',
        'AMT_CREDIT_SUM_OVERDUE',
        'AMT_CREDIT_MAX_OVERDUE',
    ],
)

bureau_balance = load_table(
    'bureau_balance',
    usecols=['SK_ID_BUREAU', 'STATUS'],
)

status_severity = {
    'X': 0,
    'C': 0,
    '0': 0,
    '1': 1,
    '2': 2,
    '3': 3,
    '4': 4,
    '5': 5,
}

bureau_balance['STATUS_SEVERITY'] = (
    bureau_balance['STATUS'].map(status_severity).fillna(0).astype('int8')
)

bureau_balance_status = (
    bureau_balance.groupby('SK_ID_BUREAU', as_index=False)['STATUS_SEVERITY']
    .max()
    .rename(columns={'STATUS_SEVERITY': 'BUREAU_BALANCE_STATUS_MAX'})
)

bureau_with_status = bureau.merge(
    bureau_balance_status,
    on='SK_ID_BUREAU',
    how='left',
)

bureau_with_status.head()


In [ ]:
# Customer-level bureau overdue aggregation.
bureau_agg = bureau_with_status.groupby('SK_ID_CURR').agg(
    {
        'CREDIT_DAY_OVERDUE': ['max'],
        'AMT_CREDIT_SUM_OVERDUE': ['sum'],
        'AMT_CREDIT_MAX_OVERDUE': ['max'],
        'BUREAU_BALANCE_STATUS_MAX': ['max'],
    }
)

bureau_agg = flatten_groupby_result(bureau_agg, 'BUREAU_')
bureau_agg.head()


In [ ]:
# Customer-level POS/CASH delinquency aggregation.
pos_cash = load_table(
    'pos_cash_balance',
    usecols=['SK_ID_CURR', 'SK_DPD', 'SK_DPD_DEF'],
)

pos_cash_agg = pos_cash.groupby('SK_ID_CURR').agg(
    {
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean'],
    }
)

pos_cash_agg = flatten_groupby_result(pos_cash_agg, 'POS_')
pos_cash_agg.head()


In [ ]:
# Credit-card minimum-payment gap feature and customer-level max aggregation.
credit_card = load_table(
    'credit_card_balance',
    usecols=['SK_ID_CURR', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT'],
)

credit_card['CC_MIN_PAYMENT_GAP'] = (
    credit_card['AMT_INST_MIN_REGULARITY'] - credit_card['AMT_PAYMENT_CURRENT']
).clip(lower=0)

credit_card_agg = credit_card.groupby('SK_ID_CURR').agg(
    {
        'CC_MIN_PAYMENT_GAP': ['max'],
    }
)

credit_card_agg = flatten_groupby_result(credit_card_agg, 'CC_')
credit_card_agg.head()


In [ ]:
# Installment underpayment feature and customer-level max aggregation.
installments = load_table(
    'installments_payments',
    usecols=['SK_ID_CURR', 'AMT_INSTALMENT', 'AMT_PAYMENT'],
)

installments['PAYMENT_DIFF'] = (
    installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
)

installments_agg = installments.groupby('SK_ID_CURR').agg(
    {
        'PAYMENT_DIFF': ['max'],
    }
)

installments_agg = flatten_groupby_result(installments_agg, 'INS_')
installments_agg.head()


In [ ]:
# Left-join all aggregate feature tables back to the application table.
for features in [bureau_agg, pos_cash_agg, credit_card_agg, installments_agg]:
    app_all = app_all.merge(features, on='SK_ID_CURR', how='left')

app_train_fe = app_all.loc[app_all['_split'] == 'train'].drop(columns=['_split']).copy()
app_test_fe = app_all.loc[app_all['_split'] == 'test'].drop(columns=['_split', 'TARGET']).copy()

app_train_fe.shape, app_test_fe.shape
